In [1]:
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoProcessor, AutoModel, Qwen2ForCausalLM

/home/y/wsl_code/qwiglip_vlm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

#load models
LLM_NAME = "Qwen/Qwen2-0.5B-Instruct"
VISION_NAME = "google/siglip-base-patch16-224"

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
processor = AutoProcessor.from_pretrained(VISION_NAME)

tokenizer.pad_token = tokenizer.eos_token

special_tokens = {"additional_special_tokens": ["<image>"]}
tokenizer.add_special_tokens(special_tokens)
IMAGE_TOKEN_ID = tokenizer.convert_tokens_to_ids("<image>")

vision_model = AutoModel.from_pretrained(VISION_NAME)
llm = Qwen2ForCausalLM.from_pretrained(LLM_NAME)

llm.resize_token_embeddings(len(tokenizer))

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 884.29it/s]


Embedding(151647, 896)

In [3]:
#check image token number
image = Image.open("images/000000391895.jpg").convert("RGB")
inputs = processor(images=image, return_tensors="pt")

with torch.no_grad():
    outputs = vision_model.vision_model(pixel_values=inputs["pixel_values"])

print(outputs.last_hidden_state.shape)


torch.Size([1, 196, 768])


In [4]:
#create special image token
special_tokens = {"additional_special_tokens": ["<image>"]}
tokenizer.add_special_tokens(special_tokens)

IMAGE_TOKEN_ID = tokenizer.convert_tokens_to_ids("<image>")

print(IMAGE_TOKEN_ID)
print(tokenizer.convert_ids_to_tokens(IMAGE_TOKEN_ID))

151646
<image>


In [5]:
#function to convert messages into string
NUM_IMAGE_TOKENS = 196

def format_chat_with_image_tokens(messages, num_image_tokens):
    image_block = " ".join(["<image>"] * num_image_tokens)

    text = ""
    for msg in messages:
        role = msg["role"]
        content = msg["content"]

        if role == "user":
            content = content.replace("<image>", image_block)
            text += f"USER: {content}\n"
        elif role == "assistant":
            text += f"ASSISTANT: {content}\n"

    return text

In [6]:
#label masking only train on assistant response
def create_labels(input_ids, text, tokenizer):
    labels = input_ids.clone()

    #seach starting position of assistant response
    assistant_prefix = "ASSISTANT:"
    assistant_start = text.find(assistant_prefix)

    #tokenize prefix
    prefix = text[:assistant_start + len(assistant_prefix)]
    prefix_ids = tokenizer(prefix, return_tensors="pt", add_special_tokens=False).input_ids[0]

    prefix_len = len(prefix_ids)

    #mask prefix tokens
    labels[:prefix_len] = -100

    return labels

In [7]:
#image loader

def load_image(path):
    return Image.open(path).convert("RGB")

In [8]:
import torch

#collate function to process batch of data
def collate_fn(batch):

    images = []
    texts = []

    for sample in batch:

        #load image
        image = load_image(sample["image_path"])
        images.append(image)

        #format chat
        text = format_chat_with_image_tokens(sample["messages"], NUM_IMAGE_TOKENS)
        texts.append(text)

    #process images
    pixel_values = processor(images=images, return_tensors="pt")["pixel_values"]

    #tokenize text
    tokenized = tokenizer(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt",
        add_special_tokens=True
    )

    input_ids = tokenized["input_ids"]
    attention_mask = tokenized["attention_mask"]

    #create labels
    labels = []

    for i in range(len(texts)):
        label = create_labels(input_ids[i], texts[i], tokenizer)
        labels.append(label)

    labels = torch.stack(labels)
    image_token_mask = (input_ids == IMAGE_TOKEN_ID)

    return {
        "pixel_values": pixel_values,
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "image_token_mask": image_token_mask,
        "texts": texts,
    }

In [9]:
from torch.utils.data import DataLoader
from datasets import load_from_disk

#load chat dataset
chat_dataset = load_from_disk("coco_chat_dataset")

loader = DataLoader(chat_dataset, batch_size=2, collate_fn=collate_fn)

batch = next(iter(loader))

print(batch["pixel_values"].shape)
print(batch["input_ids"].shape)
print(batch["labels"].shape)
print(batch["image_token_mask"].shape)
print(batch["image_token_mask"][0].sum())
print(batch["texts"][0])


torch.Size([2, 3, 224, 224])
torch.Size([2, 417])
torch.Size([2, 417])
torch.Size([2, 417])
tensor(196)
USER: <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <i

In [10]:
#sanity check
print(tokenizer.decode(batch["input_ids"][0]))
print(batch["labels"][0])

USER: <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <image> <i

In [11]:
#define projector

class MLPProjector(nn.Module):
    def __init__(self, vision_hidden_size, llm_hidden_size, dropout=0.1):
        super().__init__()
        self.pre_norm = nn.LayerNorm(vision_hidden_size)
        self.net = nn.Sequential(
            nn.Linear(vision_hidden_size, llm_hidden_size),
            nn.GELU(),
            nn.Linear(llm_hidden_size, llm_hidden_size),
            nn.Dropout(dropout),
        )
        self.post_norm = nn.LayerNorm(llm_hidden_size)
    
    def forward(self, x):
        x = self.pre_norm(x)
        x = self.net(x)
        x = self.post_norm(x)
        return x

In [12]:
#define VLM model

class SiglipQwenVLM(nn.Module):
    def __init__(self, vision_model, llm, image_token_id, dropout=0.1):
        super().__init__()

        self.vision_model = vision_model
        self.llm = llm
        self.image_token_id = image_token_id

        vision_hidden_size = self.vision_model.config.vision_config.hidden_size
        llm_hidden_size = self.llm.config.hidden_size

        self.projector = MLPProjector(
            vision_hidden_size=vision_hidden_size,
            llm_hidden_size=llm_hidden_size,
            dropout=dropout
        )

    def forward(self, pixel_values, input_ids, attention_mask=None, labels=None):
        #encode image into patch tokens
        vision_outputs = self.vision_model.vision_model(pixel_values=pixel_values)
        image_features = vision_outputs.last_hidden_state #[B, no image patches, hidden(vm)]

        #project image features to LLM space
        projected_image_features = self.projector(image_features) #[B, no image patches, hidden(llm)]

        #get normal text embeddings from LLM
        inputs_embeds = self.llm.get_input_embeddings()(input_ids) #[B, seq_len, hidden(llm)]

        #matching dtype
        projected_image_features = projected_image_features.to(
            dtype=inputs_embeds.dtype,
            device=inputs_embeds.device
        )

        batch_size = input_ids.size(0)
        num_image_tokens = projected_image_features.size(1)

        #replace each <image> with one projected image token
        for b in range(batch_size):
            image_positions = torch.where(input_ids[b] == self.image_token_id)[0]

            if len(image_positions) != num_image_tokens:
                raise ValueError(
                    f"Sample {b}: found {len(image_positions)} <image> tokens, "
                    f"but vision encoder produced {num_image_tokens} image tokens."
                )
            
            inputs_embeds[b, image_positions, :] = projected_image_features[b]
        
        #run llm using inputs_embeds
        outputs = self.llm(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=labels,
        )

        return outputs

In [13]:
#declare model
model = SiglipQwenVLM(
    vision_model=vision_model,
    llm=llm,
    image_token_id=IMAGE_TOKEN_ID,
    dropout=0.1
).to(DEVICE)

In [14]:
#freeze vision model
for param in model.vision_model.parameters():
    param.requires_grad = False

for param in model.llm.parameters():
    param.requires_grad = False

for param in model.projector.parameters():
    param.requires_grad = True


In [15]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Trainable params: {trainable:,}")
print(f"Total params: {total:,}")
print(f"Percent trainable: {100 * trainable / total:.4f}%")

Trainable params: 1,496,064
Total params: 698,425,858
Percent trainable: 0.2142%


In [16]:
#forward pass sanity check
batch = {
    k: v.to(DEVICE) if torch.is_tensor(v) else v
    for k, v in batch.items()
    if k != "texts"
}

with torch.no_grad():
    outputs = model(
        pixel_values=batch["pixel_values"],
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        labels=batch["labels"],
    )
print(outputs.loss)
print(outputs.logits.shape)

tensor(4.1901, device='cuda:0')
torch.Size([2, 417, 151647])


In [17]:
#backward pass sanity check
model.train()
model.zero_grad()

outputs = model(
    pixel_values=batch["pixel_values"],
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
    labels=batch["labels"],
)

loss = outputs.loss
print(f"Loss: {loss.item()}")

loss.backward()

print("Projector grad exists:", model.projector.net[0].weight.grad is not None)
print("Projector grad norm:", model.projector.net[0].weight.grad.norm().item())

Loss: 4.246748447418213
Projector grad exists: True
Projector grad norm: 11.758282661437988


In [18]:
from sklearn.model_selection import train_test_split

dataset = load_from_disk("coco_chat_dataset")

#train test split
dataset = dataset.train_test_split(test_size = 0.05, seed=42)

train_dataset = dataset["train"]
val_dataset = dataset["test"]

train_dataset[0]


{'messages': [{'content': '<image>\nDescribe the image.', 'role': 'user'},
  {'content': 'A street clock on a brick sidewalk next to a large building.',
   'role': 'assistant'}],
 'image_path': 'images/000000387431.jpg'}

In [19]:
#define dataloader

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_fn,
)

In [20]:
import torch.optim as optim

#define optimizer
optimizer = optim.AdamW(
    model.projector.parameters(),
    lr=1e-4,
    weight_decay=0.01,
)

In [21]:
from tqdm import tqdm

#validation function
def evaluate(model, val_loader):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):

            batch = {
                k: v.to(DEVICE) if torch.is_tensor(v) else v
                for k, v in batch.items()
                if k != "texts"
            }

            outputs = model(
                pixel_values=batch["pixel_values"],
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
            )

            total_loss += outputs.loss.item()

    avg_loss = total_loss / len(val_loader)
    print(f"Validation Loss: {avg_loss:.4f}")

    model.train()
    return avg_loss

In [25]:
#training loop

EPOCHS = 3
GRAD_ACCUM_STEPS = 8
MAX_GRAD_NORM = 1.0

best_val_loss = float("inf")

#record
rec_train_loss = []
rec_val_loss = []

model.train()

for epoch in range(EPOCHS):
    total_loss = 0.0
    optimizer.zero_grad()

    print("Using Device:", DEVICE)
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for step, batch in enumerate(pbar):

        batch = {
            k: v.to(DEVICE) if torch.is_tensor(v) else v
            for k, v in batch.items()
            if k != "texts"
        }
        
        outputs = model(
            pixel_values = batch["pixel_values"],
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )

        loss = outputs.loss
        loss = loss / GRAD_ACCUM_STEPS
        loss.backward()

        total_loss += loss.item()

        #gradient accumulation
        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            optimizer.zero_grad()

        pbar.set_postfix({"loss": loss.item()})

    avg_loss = total_loss / len(train_loader)
    rec_train_loss.append(avg_loss)
    print(f"Epoch {epoch+1} Train Loss: {avg_loss:.4f}")

    val_loss = evaluate(model, val_loader)
    rec_val_loss.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss    
        torch.save(model.state_dict(), "best_projector.pt")
        print("Best projector saved.")
    

Using Device: cuda


Epoch 1: 100%|██████████| 4750/4750 [31:04<00:00,  2.55it/s, loss=0.374]  


Epoch 1 Train Loss: 0.3360


Validation: 100%|██████████| 250/250 [00:35<00:00,  7.00it/s]


Validation Loss: 2.5004
Best projector saved.
Using Device: cuda


...

Validation Loss: 2.4599
Best projector saved.
Using Device: cuda


Epoch 3: 100%|██████████| 4750/4750 [30:21<00:00,  2.61it/s, loss=0.337]  


Epoch 3 Train Loss: 0.2696


Validation: 100%|██████████| 250/250 [00:20<00:00, 11.99it/s]


Validation Loss: 2.4399
Best projector saved.
